# 3.2 Mixture of experts: many parameters stored, few used per image

At the end of this notebook you:

- understand the difference between a mixture of experts (MoE) model and a dense model
- understand the difference between parameters during training and inference of a MoE model
- can reason about tradeoffs between MoE models and Dense models

It would be nice if I could really show you the scaling laws. However, to really show you scaling laws, we would need to run very long experiments with datasets into tens or hundreds of millions of items.

That is why I decided to put artificial constraints on the compute, and increase the diversity of the data. This way we can still illustrate scaling laws, without the need to run experiments for days.

1. Lets imagine a scenario where you have limited access to compute. For example, you are running a model on mobile phones.
2. Lets also imagine that we somehow have a very very diverse dataset. 

The combination of these two will surface the problem that mixture of experts models are an answer to:

you have a diverse dataset, so you NEED a complex model. But you also have constraints on your compute available (eg you dont have a GPU cluster at hand). 

In this situation, a MoE model can really help you out. 

In this lesson, we will artificially create this situation by

1. using a dataset that is a mixture of 7 datasets (this give us a very diverse dataset in need for expert models)
2. restrict the amount of compuyte used by restricting the amount of units in the linear layers (the width of the layers)

![](img/mix7-dataset.png )



The mix is one cache on the hub, `pttrn-io/mix7-dinov2-small`: every image of seven datasets
as a 384-dim vector. `dinov2-small` is a pretrained image encoder, as in lesson 1, that turns
a photo into 384 numbers; the cache keeps its CLS token. `load_mix` (in `mixture.py`, next to
this notebook) reads it with a label from the 321 classes of the whole mix, so that class 0 of
EuroSAT does not collide with class 0 of DTD, and with the domain of every row. The models
never see the domain; the notebook keeps it to read accuracy and routing per domain
afterwards.

The 64-number summary is PCA (lesson 1, `01` §1.1): the 64 directions along which the train
vectors vary most, fitted on the train rows only, and every vector projected onto them. Both
models below see the same 64 numbers. The summary is artificial: it keeps the router (the part of a mixture of experts that picks
an expert, 3.2.2) cheap.
On the full 384 numbers the router alone would be `384 · 8 + 8 = 3,080` parameters, more than
the smallest mixture of experts below uses per image in all; on 64 numbers it is 520, small next to the
experts, as a router is in large networks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from mixture import DOMAINS, load_mix, time_per_image, train
from notebooktester import param
from torch import nn
from vectormesh.components import SparseMoE

full_train = load_mix("train")  # 384-dim vectors, labels, domain of every row
full_test = load_mix("test")
K = int(full_train[1].max()) + 1

P = 64
center = full_train[0].mean(0)
_, _, Vh = torch.linalg.svd(full_train[0] - center, full_matrices=False)
V = Vh[:P].T  # (384, P): the 64 directions of largest variance
train_rows = ((full_train[0] - center) @ V, full_train[1], full_train[2])
test_rows = ((full_test[0] - center) @ V, full_test[1], full_test[2])

kept = (train_rows[0] ** 2).sum() / ((full_train[0] - center) ** 2).sum()
print(f"{K} classes; {P} numbers per image keep {kept:.0%} of the variance of the 384")
print(f"{'domain':15s} {'classes':>7s} {'train':>6s} {'test':>5s}")
for i, name in enumerate(DOMAINS):
    classes = len(np.unique(train_rows[1][train_rows[2] == i]))
    n_train, n_test = (train_rows[2] == i).sum(), (test_rows[2] == i).sum()
    print(f"{name:15s} {classes:7d} {n_train:6d} {n_test:5d}")
print(f"{'all':15s} {K:7d} {len(train_rows[1]):6d} {len(test_rows[1]):5d}")

## 3.2.2 Two heads, counted

**The dense head** is lesson 1's head with one hidden layer of width `w`:
`(B, 64)` → `Linear(64, w)` → GELU → `Linear(w, 321)` → `(B, 321)`, one score per class.
It has `64·w + w` parameters in the first layer and `w·321 + 321` in the second, and every
image passes through all of them.

**The sparse mixture of experts** (sparse: only some experts run per input) holds 8 experts,
each a dense head of width `w`, and a **router**: a `Linear(64, 8)` that gives every image 8
scores, turned into 8 probabilities by a softmax. Only the expert with the highest
probability runs on the image (**top-1**), and its 321 scores are multiplied by that
probability. Choosing the top expert is an argmax, which has no gradient; the probability
that multiplies the chosen expert's output has one, and through it the router learns which
expert to choose. In code it is `SparseMoE(experts, 64, 321, k=1)`: the experts, the input
size, the output size and `k`, the number of experts that run per image.

<svg width="720" height="300" viewBox="0 0 720 300" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="300" fill="white"/>
<defs><marker id="m1" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="22" font-weight="bold" fill="#222">dense head</text>
<text x="10" y="58" fill="#222">image</text>
<line x1="52" y1="54" x2="158" y2="54" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="105" y="46" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<rect x="160" y="32" width="250" height="44" rx="8" fill="#cfe3f3" stroke="#555"/>
<text x="285" y="52" text-anchor="middle" fill="#222">Linear(64, w), GELU, Linear(w, 321)</text>
<text x="285" y="68" text-anchor="middle" font-size="11" fill="#222">every parameter used for every image</text>
<line x1="410" y1="54" x2="578" y2="54" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="494" y="46" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 321)</text>
<text x="584" y="58" fill="#222">scores</text>
<text x="10" y="112" font-weight="bold" fill="#222">sparse mixture of experts, top-1</text>
<text x="10" y="208" fill="#222">image</text>
<line x1="52" y1="204" x2="110" y2="204" stroke="#444" stroke-width="1.5"/>
<path d="M 110 204 L 110 150 L 158 150" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<path d="M 110 204 L 110 240 L 298 240" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="52" y="222" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<rect x="160" y="128" width="150" height="44" rx="8" fill="#f6d9a8" stroke="#555"/>
<text x="235" y="148" text-anchor="middle" fill="#222">router</text>
<text x="235" y="164" text-anchor="middle" font-size="11" fill="#222">Linear(64, 8), softmax</text>
<line x1="310" y1="150" x2="388" y2="150" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="349" y="142" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 8)</text>
<text x="394" y="146" fill="#222">top-1: expert e,</text>
<text x="394" y="162" fill="#222">probability p</text>
<path d="M 450 168 L 450 196" fill="none" stroke="#444" stroke-width="1.5" stroke-dasharray="4 3" marker-end="url(#m1)"/>
<text x="458" y="188" font-size="11" fill="#222">picks the expert</text>
<rect x="300" y="200" width="300" height="80" rx="8" fill="none" stroke="#999" stroke-dasharray="4 3"/>
<rect x="310" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="346" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="382" y="208" width="30" height="64" rx="4" fill="#cfe3f3" stroke="#555"/>
<rect x="418" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="454" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="490" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="526" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<rect x="562" y="208" width="30" height="64" rx="4" fill="#eeeeee" stroke="#aaa"/>
<text x="397" y="244" text-anchor="middle" font-size="11" fill="#222">e</text>
<text x="450" y="294" text-anchor="middle" font-size="11" fill="#222">8 experts, each a dense head of width w; only e runs on this image</text>
<line x1="600" y1="240" x2="638" y2="240" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="644" y="236" fill="#222">× p</text>
<text x="644" y="252" font-family="monospace" font-size="11" fill="#222">(B, 321)</text>
</svg>


The shapes, arrow by arrow:

- `(B, 64)`: a batch of `B` images, 64 numbers each, goes to the router and to the experts.
- `(B, 8)`: the router's 8 probabilities per image; the largest names the expert `e` that
  runs, and its value is the `p` that scales that expert's output.
- `(B, 321)`: the rows sent to expert `e` are gathered into one smaller batch, the expert
  gives 321 scores per row, and the scores, times `p`, are scattered back to their place in
  the batch.

So the MoE has two counts. **Stored**: the router and all 8 experts, everything that is
trained and kept. **Used** for one image: the router and one expert; the other 7 experts
take no part in that image. A used parameter is one multiply and one add per image, so the
used count measures arithmetic, not time; 3.2.6 asks whether the clock agrees.

Why experts could help under a tight budget: a dense head of width 6 has a hidden layer of 6
numbers to separate all 321 classes; an expert only has to separate the classes the router
sends it.

**The control.** The budget is on the parameters used per image. A MoE that beats a
smaller dense head could win only because it uses more, so each MoE is compared with the
dense head whose used parameters come closest to its own, router included. That pair
differs in one factor: what the MoE stores beyond what it uses, seven more experts to choose
from. A comparison that looks just as natural is the dense head as wide as one expert, "one
expert alone". It leaves out the router's parameters, so it flatters the MoE; 3.2.4 shows by
how much.

In [ ]:
N_EXPERTS = 8
WIDTHS = [4, 8, 16]  # the width of one expert


def dense(width: int, dim: int = P) -> nn.Module:
    return nn.Sequential(nn.Linear(dim, width), nn.GELU(), nn.Linear(width, K))


def moe(width: int, dim: int = P) -> nn.Module:
    return SparseMoE([dense(width, dim) for _ in range(N_EXPERTS)], dim, K, k=1)


def stored(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def used(model: nn.Module) -> int:
    # The parameters one image passes through: router and one expert for a MoE.
    if isinstance(model, SparseMoE):
        return stored(model.router) + stored(model.experts[0])
    return stored(model)


def control_width(width: int, dim: int = P) -> int:
    # A dense head of width d has d * (dim + 1 + K) + K parameters; take the d closest
    # to the used parameters of moe(width, dim).
    return round((used(moe(width, dim)) - K) / (dim + 1 + K))


CONTROL = {w: control_width(w) for w in WIDTHS}
DENSE_WIDTHS = sorted({24, *WIDTHS, *CONTROL.values()})

print(f"router: {stored(moe(4).router)} parameters\n")
print(f"{'model':>10s} {'stored':>8s} {'used per image':>15s}")
for w in WIDTHS:
    for model, name in [
        (moe(w), f"MoE 8 x {w}"),
        (dense(CONTROL[w]), f"dense {CONTROL[w]}"),
    ]:
        print(f"{name:>10s} {stored(model):8,d} {used(model):15,d}")
print(f"\ndense widths trained: {DENSE_WIDTHS}")

Each MoE stores 6.5 to 7.5 times what it uses. The router is `64 · 8 + 8 = 520` parameters,
an eighth of what the MoE `8 × 8` uses. Width 24 is trained as the top of the dense curve.

## 3.2.3 The experiment

**Question.** At the same parameters used per image, does the MoE classify the mix better
than a dense head?

**Setup.** The dense heads of every width and the three MoEs, one seed each, all trained the
same way for 100 epochs (`train` in `mixture.py`, whose docstring gives the recipe); the epoch
with the best validation accuracy is kept and the test split is scored once. The run records
the test accuracy, the expert the router picks for every test image, and the time per image
that 3.2.6 reads.

In [ ]:
SEED = 0
EPOCHS = param(100, test=1)
CALLS = param(200, test=5)  # timed forward passes per batch size

X_test, y_test, domain_test = test_rows


def run(name: str, make) -> dict:
    model, X_std, pred, valid_curve = train(make, train_rows, X_test, SEED, EPOCHS)
    right = (pred == y_test).numpy()
    with torch.no_grad():
        route = (
            model.router(X_std).argmax(1).numpy()
            if isinstance(model, SparseMoE)
            else None
        )
    return {
        "name": name,
        "used": used(model),
        "test": right.mean(),
        "valid_curve": valid_curve,
        "route": route,
        "X_std": X_std,
        "us_batch1": time_per_image(model, 1, P, CALLS),
        "us_batch256": time_per_image(model, 256, P, CALLS),
    }


dense_runs = {d: run(f"dense {d}", lambda d=d: dense(d)) for d in DENSE_WIDTHS}
moe_runs = {w: run(f"MoE 8 x {w}", lambda w=w: moe(w)) for w in WIDTHS}

print(f"{'model':>10s} {'used':>6s} {'kept epoch':>10s} {'test accuracy':>14s}")
for r in [*dense_runs.values(), *moe_runs.values()]:
    kept = int(np.argmax(r["valid_curve"])) + 1
    print(f"{r['name']:>10s} {r['used']:6,d} {kept:10d} {r['test']:14.3f}")

**Has every run settled?** A model scored while it was still learning would measure its
budget of epochs, not its architecture. The validation accuracy per epoch shows where each run
stands at the end; the dot marks the kept epoch.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
epochs = np.arange(1, EPOCHS + 1)
for a, runs, title in [
    (axs[0], dense_runs, "dense heads"),
    (axs[1], moe_runs, "MoE, 8 experts"),
]:
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(runs)))
    for r, color in zip(runs.values(), colors):
        best = int(np.argmax(r["valid_curve"]))
        a.plot(epochs, r["valid_curve"], color=color, lw=1.2, label=r["name"])
        a.plot(best + 1, r["valid_curve"][best], "o", color=color)
    a.set_title(title, fontsize=10)
    a.set_xlabel("epoch")
    a.grid(alpha=0.3)
    a.legend(frameon=False, fontsize=7, ncol=2, loc="lower right")
axs[0].set_ylabel("validation accuracy")
fig.suptitle("Every run has flattened well before epoch 100")
plt.tight_layout()
plt.show()

**Reading.** Every run, dense and MoE, is flat over its last 40 epochs; the kept epoch, between
49 and 97, is where the flat curve happens to peak. The 100 epochs measure the architecture,
not the budget of epochs.

## 3.2.4 The MoE against the dense curve

The test accuracy of every run against the parameters it uses per image: the dense heads as
one curve, the MoEs as points.

In [ ]:
fig, a = plt.subplots(figsize=(7.5, 4.2))
xd = [r["used"] for r in dense_runs.values()]
yd = [r["test"] for r in dense_runs.values()]
a.plot(xd, yd, "o-", color="#0072B2", label="dense head")
for d, x, y in zip(dense_runs, xd, yd):
    a.annotate(
        str(d),
        (x, y),
        textcoords="offset points",
        xytext=(0, 7),
        ha="center",
        fontsize=8,
        color="#0072B2",
    )
for w, r in moe_runs.items():
    a.plot(
        r["used"],
        r["test"],
        "s",
        color="#D55E00",
        ms=8,
        label="MoE, 8 experts, top-1" if w == WIDTHS[0] else None,
    )
    a.annotate(
        f"8 x {w}",
        (r["used"], r["test"]),
        textcoords="offset points",
        xytext=(8, 0),
        ha="left",
        va="center",
        fontsize=8,
        color="#D55E00",
    )
a.set_xscale("log")
a.set_xticks(xd, [f"{x:,}" for x in xd], fontsize=7, rotation=45)
a.minorticks_off()
a.set_ylim(min(yd) - 0.05, max(yd) + 0.05)
a.set_xlabel("parameters used per image (log); dense width above each point")
a.set_ylabel("test accuracy, 7 domains")
a.set_title("At all three budgets the MoE lies above the dense curve")
a.grid(alpha=0.3)
a.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

print(
    f"{'MoE':>9s} {'used':>6s} {'test':>6s} | {'control: same used':>24s} | {'one expert alone':>24s}"
)
for w, r in moe_runs.items():
    one, same = dense_runs[w], dense_runs[CONTROL[w]]
    print(
        f"{r['name']:>9s} {r['used']:6,d} {r['test']:6.3f} | "
        f"{same['name']:>9s} {same['used']:5,d} {same['test']:.3f} {100 * (r['test'] - same['test']):+5.1f} | "
        f"{one['name']:>9s} {one['used']:5,d} {one['test']:.3f} {100 * (r['test'] - one['test']):+5.1f}"
    )

**Reading.** At all three budgets the MoE lies above the dense curve. Against its control,
the dense head with the same used parameters, the MoE `8 × 4` gains 15.4 points (0.514 against
0.360 for dense 5), the MoE `8 × 8` 13.1 points (0.656 against 0.525 for dense 9) and the MoE
`8 × 16` 6.7 points (0.726 against 0.660 for dense 17). The gain shrinks as the budget grows,
but even dense 24, which uses 9,585 parameters per image, stays below the MoE `8 × 16` at
0.702. With 321 classes the dense curve is still climbing at width 24: the budget is the
bottleneck at every width trained here, and that is where experts help.

"One expert alone" flatters the MoE: against dense 4, 8 and 16 it gains 23.3, 15.9 and 7.7
points, because those heads leave out the router's 520 parameters. The control is the head
with the same used parameters.

This is one seed; another seed could move each number by a point or two. That does not reach
gains of 7 to 15 points.

## 3.2.5 What the router learned

**Question.** The router never saw which domain an image comes from. Does it still send each
domain to experts of its own?

**Setup.** For the MoE `8 × 8`: the share of each domain's test images that the router sends
to each expert (cells below 0.05 are left blank), and per class the share of its test images
that go to the one expert it is sent to most, averaged over the 321 classes; that share is 1
when all images of a class go to the same expert. The vectors of a class lie close together, so even an untrained router sends a class's
images partly to one expert; both are therefore also computed for the untrained router of the
same seed. Last, for each
expert, the number of classes whose images it gets most of.

In [ ]:
def shares(route: np.ndarray) -> np.ndarray:
    # Row i: the share of domain i's test images routed to each expert.
    return np.array(
        [
            np.bincount(route[domain_test == i], minlength=N_EXPERTS)
            / (domain_test == i).sum()
            for i in range(len(DOMAINS))
        ]
    )


trained = moe_runs[8]
torch.manual_seed(SEED)
with torch.no_grad():
    untrained_route = moe(8).router(trained["X_std"]).argmax(1).numpy()

fig, axs = plt.subplots(1, 2, figsize=(13, 4.4))
for a, (title, route) in zip(
    axs, [("untrained router", untrained_route), ("trained router", trained["route"])]
):
    table = shares(route)
    a.imshow(table, cmap="Greys", vmin=0, vmax=1)
    for i in range(len(DOMAINS)):
        for e in range(N_EXPERTS):
            if table[i, e] >= 0.05:
                a.text(
                    e,
                    i,
                    f"{table[i, e]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if table[i, e] > 0.5 else "black",
                )
    a.set_yticks(range(len(DOMAINS)), DOMAINS)
    a.set_xticks(range(N_EXPERTS))
    a.set_xlabel("expert")
    a.set_title(f"MoE 8 x 8: {title}", fontsize=10)
fig.suptitle("Share of each domain's test images sent to each expert")
plt.tight_layout()
plt.show()


def class_share(route: np.ndarray) -> float:
    # Per class, the share of its test images sent to its most used expert; mean over classes.
    y = y_test.numpy()
    return float(
        np.mean(
            [
                np.bincount(route[y == k], minlength=N_EXPERTS).max() / (y == k).sum()
                for k in np.unique(y)
            ]
        )
    )


def classes_per_expert(route: np.ndarray) -> np.ndarray:
    # For each expert, the number of classes whose test images it gets most of.
    y = y_test.numpy()
    home = [
        np.bincount(route[y == k], minlength=N_EXPERTS).argmax() for k in np.unique(y)
    ]
    return np.bincount(home, minlength=N_EXPERTS)


print(
    f"share of a class's images at its most used expert: untrained {class_share(untrained_route):.2f}, "
    f"trained {class_share(trained['route']):.2f}"
)
y = y_test.numpy()
print("per domain, untrained -> trained:")
for i, name in enumerate(DOMAINS):
    mask = domain_test == i
    shares_d = [
        np.mean(
            [
                np.bincount(r[mask & (y == k)], minlength=N_EXPERTS).max()
                / (mask & (y == k)).sum()
                for k in np.unique(y[mask])
            ]
        )
        for r in (untrained_route, trained["route"])
    ]
    print(f"  {name:15s} {shares_d[0]:.2f} -> {shares_d[1]:.2f}")
counts = classes_per_expert(trained["route"])
print(
    f"classes per expert (most of their images): {' '.join(map(str, counts))} (sum {counts.sum()})"
)

**Reading.** The vectors are not spread evenly, so even the untrained router sends a class's
images to its most used expert 52% of the time. Training strengthens that split to 92%, most
where the untrained router split least: GTSRB goes from 0.36 to 1.00, Galaxy10 from 0.34 to
1.00 and BloodMNIST from 0.45 to 1.00; DTD moves least, from 0.53 to 0.72.
Three domains get an expert of their own: Galaxy10 goes entirely to expert 1, Flowers mostly
to expert 6 (83%) and Food-101 mostly to expert 2 (69%). The others share: BloodMNIST goes
entirely to expert 5, and two thirds of GTSRB with it; DTD and EuroSAT both go mostly to
expert 7. Each expert gets most of the images of 7 to 95 classes, where the control's hidden
layer has to separate all 321.

The router is never given the domain, nor which classes go to which expert. The slides showed
the opposite outcome, router collapse: the router sends nearly every image to a few experts,
which then learn faster and are chosen even more. Shazeer et al. (2017, *Outrageously Large
Neural Networks*) describe it and add a loss that spreads the images over the experts; the
Switch Transformer (Fedus et al., 2022) uses a simpler form of that load-balancing loss. Here,
with eight experts and seven domains, every expert gets classes of its own.


**A limit of this setup.** This notebook does not ask whether a router has to be learned, or
how much a good split is worth: with this little data and a compute budget restricted on
purpose, the experts are too small and see too few images for a test of the router to say
anything about routers in general. That question is studied on large models and datasets.
There a split pays off as long as there is a pattern in what each expert gets to see; it does
not even have to be learned: in large language models a fixed split by a hash of the token
comes close to a learned router (Roller et al., 2021, *Hash Layers for Large Sparse Models*).

## 3.2.6 What does the clock say at this scale?

The slides gave the rule for large models: compute follows the used parameters. Mixtral
stores 47B parameters and uses 13B per token, so it runs about as fast as a dense model of
13B. That rule holds when the arithmetic of an expert is most of the work. The experts here
hold a few thousand parameters.

**Question.** A MoE and its control use as many parameters per image, so as much arithmetic.
With experts this small, does the MoE answer as fast?

**Setup.** The runs of 3.2.3 timed every trained model on one CPU thread, on random vectors
of the same shape, at two batch sizes: batch 1, one photo per call, as an app that answers one
request while the user waits, and batch 256, a batch of the stream. Vectormesh checks tensor
shapes on every call while you design a model; the timing turns that check off, as a deployed
model runs without it.

In [ ]:
print(f"{'model':>10s} {'used':>6s} {'µs per image, batch 1':>22s} {'batch 256':>10s}")
for w in WIDTHS:
    for r in (dense_runs[CONTROL[w]], moe_runs[w]):
        print(
            f"{r['name']:>10s} {r['used']:6,d} {r['us_batch1']:22.1f} {r['us_batch256']:10.2f}"
        )

**Reading.** Not at this scale: at batch 1 every MoE takes 28 to 31 µs per image against 4 to 6 µs for its
control, five to seven times as long, and at batch 256 0.58 to 0.67 µs against about 0.1.
The MoE's time hardly moves with the width of its experts, so it is not their arithmetic but
the work around it: choosing the expert, sorting, gathering and scattering the rows, and a
separate matrix product for every expert that runs (one at batch 1, all eight at batch 256).
The dense head gets cheaper per image in a batch too, because one large matrix product over
many rows uses the processor better than many small ones.

**Why this differs from the slides.** The cost of routing does not grow with the experts;
the arithmetic of an expert does. With experts as small as these the routing is most of the
work, so the MoE is slower than the dense head of the same used parameters. That is a
limit of this notebook, not a property of a MoE: the wider the experts, the smaller the
share of the routing, and with the experts of a large language model a MoE costs about what
a dense model of its used parameters costs. There the accuracy gain at the same arithmetic
of 3.2.4 is a gain at the same time; that is why the budget here is simulated. These are times on one CPU thread of this machine; another machine, or a GPU,
moves them.


## What you now know

- **The control** of a MoE is the dense head that uses as many parameters per image, router
  included; against it the MoE gained 7 to 15 points, most where the budget was tightest, and the head
  as wide as one expert, without the router, overstates that gain.
- **Stored and used** are two counts: a MoE stores all its experts and uses the router and one
  of them per image, and compute follows the used ones once the experts are large; with experts as
  small as these, the routing costs more than the arithmetic it saves.
- **The experts specialise**: the router is never given the domain; training strengthens the
  split the vectors already have (52% of a class's images at one expert untrained, 92%
  trained), so each expert separates only part of the 321 classes.

## Exercises

**E1: top-2.** Route every image to its two best experts
(`SparseMoE(experts, 64, 321, k=2, normalize=True)`; `normalize=True` rescales the two router
probabilities to sum to 1). Count the parameters it uses per image (the router and two
experts), find the dense head that uses as many, and train both at `w = 8`. Does a second
expert per image buy more than the same parameters as width?

**E2: the size of the summary.** Set `P` to 32 and to 128 and rerun 3.2.3 for the MoE `8 × 8`
and its control. How does the router's share of the used parameters change, and does the
gain at the same used parameters survive?

**E3: the control with as many stored parameters.** Train the dense head that **stores** as
much as the MoE `8 × 8`, which uses several times as many parameters per image. How much more
accurate is it, how much slower at batch 256, and which would you put in the service?